# Metaclonotype API Examples

This notebook demonstrates how to build and analyze metaclonotypes as a lightweight clustering layer over `LocusRepertoire`.

In [1]:
# Print key environment versions for reproducibility
import platform
import polars as pl
import mir

print('python', platform.python_version())
print('polars', pl.__version__)
print('mir', getattr(mir, '__version__', 'unknown'))

python 3.12.12
polars 1.40.1
mir unknown


In [2]:
# Build a tiny repertoire and connected-component metaclonotypes
from mir.common.clonotype import Clonotype
from mir.common.repertoire import LocusRepertoire
from mir.common.metaclonotype import (
    metaclonotypes_from_components,
    summarize_metaclonotypes,
    functional_diversity,
    functional_hill_curve,
)

rep = LocusRepertoire(
    [
        Clonotype(sequence_id='c1', locus='TRB', junction_aa='CASSLGQETQYF', v_gene='TRBV5-1*01', j_gene='TRBJ2-7*01', duplicate_count=10, umi_count=4, _validate=False),
        Clonotype(sequence_id='c2', locus='TRB', junction_aa='CASSLGQETQFF', v_gene='TRBV5-1*01', j_gene='TRBJ2-7*01', duplicate_count=5, umi_count=3, _validate=False),
        Clonotype(sequence_id='c3', locus='TRB', junction_aa='CATSLGQETQYF', v_gene='TRBV5-1*01', j_gene='TRBJ2-7*01', duplicate_count=2, umi_count=1, _validate=False),
    ],
    locus='TRB',
)

meta = metaclonotypes_from_components([["c1", "c2"], ["c3"]])
rep.set_metaclonotypes(meta)

summary = summarize_metaclonotypes(rep, meta)
summary

/Users/mikesh/vcs/mirpy/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


cluster_id,n_members,duplicate_count,umi_count,representative_clonotype_id,representative_junction_aa,representative_v_gene,representative_j_gene
str,u32,i64,i64,str,str,str,str
"""mc_0""",2,15,7,"""c1""","""CASSLGQETQYF""","""TRBV5-1*01""","""TRBJ2-7*01"""
"""mc_1""",1,2,1,"""c3""","""CATSLGQETQYF""","""TRBV5-1*01""","""TRBJ2-7*01"""


In [3]:
# Compute functional diversity metrics from metaclonotype counts
func_div = functional_diversity(rep, meta, count_field='duplicate_count')
print(func_div)

functional_hill_curve(rep, meta, count_field='duplicate_count', q_values=[0.0, 1.0, 2.0])

DiversitySummary(abundance=17, diversity=2, singletons=0, doubletons=1, expanded=2, hyperexpanded=2, chao1=2.0, gini_simpson=0.20761245674740492, shannon=0.362210557135449)


q,hill
f64,f64
0.0,2.0
1.0,1.436501
2.0,1.262009


In [4]:
# Build metaclonotypes from ALICE/TCRNET metadata-marked representatives
from mir.biomarkers.alice import metaclonotypes_from_alice
from mir.biomarkers.tcrnet import metaclonotypes_from_tcrnet

rep.clonotypes[0].clone_metadata['alice_q_value'] = 0.01
rep.clonotypes[0].clone_metadata['tcrnet_q_value'] = 0.01

alice_meta = metaclonotypes_from_alice(rep, q_value_max=0.05)
tcrnet_meta = metaclonotypes_from_tcrnet(rep, q_value_max=0.05)

print('alice clusters', alice_meta.n_clusters)
print('tcrnet clusters', tcrnet_meta.n_clusters)

alice clusters 1
tcrnet clusters 1


## Point-by-point clustering conversions

This section maps common clustering outputs to metaclonotypes:

1. ALICE/TCRNET enriched representatives + neighbors.
2. DBSCAN/OPTICS-style label arrays from TCREmp embeddings.
3. Connected components / Leiden / Louvain from clonotype graphs.
4. Generic representative-centered search scopes.
5. Continuous-radius representative clustering.

The examples below show the lightweight label/graph conversion paths.

In [ ]:
# Convert TCREmp-style labels and graph communities to metaclonotypes
from mir.embedding import metaclonotypes_from_tcremp_labels
from mir.utils.metaclonotype_clustering import metaclonotypes_from_graph_communities
import igraph as ig

# Label-based conversion (DBSCAN/OPTICS/VDBSCAN-compatible labels)
label_meta = metaclonotypes_from_tcremp_labels(rep.clonotypes, labels=[0, 0, 1])

# Graph community conversion (components shown; Leiden/Louvain also supported)
g = ig.Graph(n=3, edges=[(0, 1)], directed=False)
g.vs['r_id'] = [c.sequence_id for c in rep.clonotypes]
g.es['weight'] = [1.0]
graph_meta = metaclonotypes_from_graph_communities(g, method='components', vertex_id_attr='r_id')

print('label clusters', label_meta.n_clusters)
print('graph clusters', graph_meta.n_clusters)